# Playground Series S6E4 - CatBoost Tuning

This notebook moves from baseline comparison into focused CatBoost experimentation. It uses stratified cross-validation, compares feature variants, tunes a compact set of CatBoost parameters, and writes a tuned submission file.

## 1. Setup

Load the Kaggle modeling stack and define shared experiment controls. The defaults are practical for iteration; increase folds or add parameter sets after the first run is stable.

In [ ]:
from pathlib import Path
from IPython.display import display

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 140)
pd.set_option("display.float_format", "{:,.5f}".format)

sns.set_theme(style="whitegrid", context="notebook")
VIRIDIS_CMAP = "viridis"
RANDOM_STATE = 42
N_SPLITS = 3
RUN_FULL_CV = True
MAKE_SUBMISSION = True

## 2. Data Loading

Load the Kaggle competition files and infer the ID and target columns from `sample_submission.csv`.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "This notebook is intended to run on Kaggle, where /kaggle/input is available."
    )


def find_file(filename: str) -> Path:
    """Return the first Kaggle input file matching the requested name.

    Args:
        filename: File name to search for below `/kaggle/input`.

    Returns:
        Path to the selected input file.
    """
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {INPUT_ROOT}. Attach the competition data."
        )
    if len(matches) > 1:
        print(f"Found multiple {filename} files. Using: {matches[0]}")
    return matches[0]


train = pd.read_csv(find_file("train.csv"))
test = pd.read_csv(find_file("test.csv"))
sample_submission = pd.read_csv(find_file("sample_submission.csv"))

target_col = sample_submission.columns[-1]
id_col = sample_submission.columns[0]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[target_col])
class_names = label_encoder.classes_.tolist()

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
print("classes:", class_names)
display(
    train[target_col]
    .value_counts(normalize=True)
    .mul(100)
    .rename("target_pct")
    .to_frame()
)

## 3. Feature Variants

Compare raw competition features against EDA-driven interaction and threshold features. The baseline notebook showed that interactions contribute, while EDA showed strong threshold behavior in soil moisture, rainfall, temperature, and wind speed.

### 3.1 Base and Interaction Features

Create the same interaction features used in the baseline notebook so tuning can compare against the previous best model.

In [ ]:
base_feature_cols = [c for c in train.columns if c not in {id_col, target_col}]

INTERACTION_PAIRS = [
    ("Crop_Growth_Stage", "Mulching_Used"),
    ("Crop_Growth_Stage", "Water_Source"),
    ("Crop_Growth_Stage", "Irrigation_Type"),
]


def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add compact EDA-driven categorical interaction features.

    Args:
        df: Input train or test frame.

    Returns:
        Copy of the input frame with interaction columns added.
    """
    out = df.copy()
    for left, right in INTERACTION_PAIRS:
        if left in out.columns and right in out.columns:
            out[f"{left}__x__{right}"] = (
                out[left].astype(str) + "__" + out[right].astype(str)
            )
    return out


train_interactions = add_interaction_features(train)
test_interactions = add_interaction_features(test)

interaction_feature_cols = [
    c for c in train_interactions.columns if c not in {id_col, target_col}
]
interaction_cols = [c for c in interaction_feature_cols if "__x__" in c]

print("Base feature count:", len(base_feature_cols))
print("Interaction feature count:", len(interaction_feature_cols))
print("Interaction columns:", interaction_cols)

### 3.2 Threshold Features

Add compact threshold indicators inspired by EDA. These may help if CatBoost benefits from explicit high-risk cut points, but they should be validated rather than assumed useful.

In [ ]:
def add_threshold_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add binary threshold indicators from EDA risk cut points.

    Args:
        df: Input feature frame.

    Returns:
        Copy of the input frame with threshold columns added.
    """
    out = df.copy()
    if "Soil_Moisture" in out.columns:
        out["Soil_Moisture_le_26"] = (out["Soil_Moisture"] <= 26.22).astype("int8")
        out["Soil_Moisture_le_20"] = (out["Soil_Moisture"] <= 20.50).astype("int8")
    if "Rainfall_mm" in out.columns:
        out["Rainfall_le_644"] = (out["Rainfall_mm"] <= 643.77).astype("int8")
    if "Temperature_C" in out.columns:
        out["Temperature_ge_30"] = (out["Temperature_C"] >= 30.21).astype("int8")
    if "Wind_Speed_kmh" in out.columns:
        out["Wind_Speed_ge_10_5"] = (out["Wind_Speed_kmh"] >= 10.48).astype("int8")
    return out


train_full = add_threshold_features(train_interactions)
test_full = add_threshold_features(test_interactions)

threshold_cols = [c for c in train_full.columns if c not in train_interactions.columns]
full_feature_cols = [c for c in train_full.columns if c not in {id_col, target_col}]

print("Threshold columns:", threshold_cols)
print("Full feature count:", len(full_feature_cols))

## 4. Evaluation Utilities

Define reusable cross-validation, metric, and plotting helpers. Macro F1 and `High` recall remain important because the `High` class is rare.

In [ ]:
def get_categorical_columns(df: pd.DataFrame, feature_cols: list[str]) -> list[str]:
    """Return feature names that CatBoost should treat as categorical."""
    return [
        c
        for c in feature_cols
        if df[c].dtype == "object" or str(df[c].dtype).startswith("category")
    ]


def make_pool(
    df: pd.DataFrame, feature_cols: list[str], target: np.ndarray | None = None
) -> Pool:
    """Build a CatBoost Pool with inferred categorical feature indices."""
    cat_cols = get_categorical_columns(df, feature_cols)
    cat_indices = [feature_cols.index(c) for c in cat_cols]
    return Pool(df[feature_cols], label=target, cat_features=cat_indices)


def evaluate_fold(
    y_true: np.ndarray, pred: np.ndarray, proba: np.ndarray
) -> dict[str, float]:
    """Calculate fold-level multiclass metrics and per-class scores."""
    row = {
        "accuracy": accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro"),
        "weighted_f1": f1_score(y_true, pred, average="weighted"),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "log_loss": log_loss(y_true, proba, labels=np.arange(len(class_names))),
    }
    report = classification_report(
        y_true, pred, target_names=class_names, output_dict=True, zero_division=0
    )
    for label in class_names:
        row[f"{label}_precision"] = report[label]["precision"]
        row[f"{label}_recall"] = report[label]["recall"]
        row[f"{label}_f1"] = report[label]["f1-score"]
    return row


def summarize_cv(results: pd.DataFrame) -> pd.DataFrame:
    """Aggregate cross-validation metrics by experiment."""
    metric_cols = [c for c in results.columns if c not in {"experiment", "fold"}]
    summary = results.groupby("experiment")[metric_cols].agg(["mean", "std"])
    return summary.sort_values(("macro_f1", "mean"), ascending=False)


def plot_confusion(y_true: np.ndarray, pred: np.ndarray, title: str) -> None:
    """Plot count and row-normalized confusion matrices."""
    cm = confusion_matrix(y_true, pred, labels=np.arange(len(class_names)))
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap=VIRIDIS_CMAP,
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0],
    )
    axes[0].set_title(f"{title} - Counts")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".3f",
        cmap=VIRIDIS_CMAP,
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[1],
    )
    axes[1].set_title(f"{title} - Row Normalized")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")
    plt.tight_layout()
    plt.show()

## 5. Experiment Grid

Start with a compact set of CatBoost configurations. The goal is not exhaustive search yet; it is to understand whether depth, regularization, class weighting, and feature variants improve over the baseline.

In [ ]:
BASE_CATBOOST_PARAMS = {
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
    "thread_count": -1,
}

class_counts = np.bincount(y)
class_weights = (len(y) / (len(class_counts) * class_counts)).tolist()
print("Class weights by encoded class:", dict(zip(class_names, class_weights)))

EXPERIMENTS = [
    {
        "experiment": "baseline_interactions",
        "train_df": train_interactions,
        "test_df": test_interactions,
        "feature_cols": interaction_feature_cols,
        "params": {
            **BASE_CATBOOST_PARAMS,
            "iterations": 600,
            "learning_rate": 0.06,
            "depth": 7,
            "l2_leaf_reg": 3.0,
        },
    },
    {
        "experiment": "deeper_interactions",
        "train_df": train_interactions,
        "test_df": test_interactions,
        "feature_cols": interaction_feature_cols,
        "params": {
            **BASE_CATBOOST_PARAMS,
            "iterations": 800,
            "learning_rate": 0.045,
            "depth": 8,
            "l2_leaf_reg": 4.0,
            "random_strength": 1.0,
        },
    },
    {
        "experiment": "regularized_interactions",
        "train_df": train_interactions,
        "test_df": test_interactions,
        "feature_cols": interaction_feature_cols,
        "params": {
            **BASE_CATBOOST_PARAMS,
            "iterations": 900,
            "learning_rate": 0.04,
            "depth": 7,
            "l2_leaf_reg": 8.0,
            "random_strength": 1.5,
            "bagging_temperature": 0.5,
        },
    },
    {
        "experiment": "weighted_interactions",
        "train_df": train_interactions,
        "test_df": test_interactions,
        "feature_cols": interaction_feature_cols,
        "params": {
            **BASE_CATBOOST_PARAMS,
            "iterations": 700,
            "learning_rate": 0.05,
            "depth": 7,
            "l2_leaf_reg": 5.0,
            "class_weights": class_weights,
        },
    },
    {
        "experiment": "threshold_features",
        "train_df": train_full,
        "test_df": test_full,
        "feature_cols": full_feature_cols,
        "params": {
            **BASE_CATBOOST_PARAMS,
            "iterations": 800,
            "learning_rate": 0.045,
            "depth": 7,
            "l2_leaf_reg": 5.0,
            "random_strength": 1.0,
        },
    },
]

pd.DataFrame(
    [
        {
            "experiment": e["experiment"],
            "n_features": len(e["feature_cols"]),
            **{
                k: v
                for k, v in e["params"].items()
                if k
                not in {
                    "loss_function",
                    "eval_metric",
                    "verbose",
                    "allow_writing_files",
                    "thread_count",
                }
            },
        }
        for e in EXPERIMENTS
    ]
)

## 6. Stratified Cross-Validation

Run stratified cross-validation for each experiment. The out-of-fold predictions are retained for diagnostics, including confusion matrices and class-level reports.

In [ ]:
cv_rows = []
oof_predictions = {}
oof_probabilities = {}
trained_fold_models = {}

if RUN_FULL_CV:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for exp in EXPERIMENTS:
        name = exp["experiment"]
        df = exp["train_df"]
        feature_cols = exp["feature_cols"]
        params = exp["params"]
        print(f"Running {name} with {len(feature_cols)} features")

        oof_pred = np.zeros(len(df), dtype=int)
        oof_proba = np.zeros((len(df), len(class_names)), dtype=float)
        fold_models = []

        for fold, (tr_idx, va_idx) in enumerate(
            skf.split(df[feature_cols], y), start=1
        ):
            train_pool = make_pool(df.iloc[tr_idx], feature_cols, y[tr_idx])
            valid_pool = make_pool(df.iloc[va_idx], feature_cols, y[va_idx])
            model = CatBoostClassifier(**params)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
            proba = model.predict_proba(valid_pool)
            pred = proba.argmax(axis=1)

            oof_pred[va_idx] = pred
            oof_proba[va_idx] = proba
            row = evaluate_fold(y[va_idx], pred, proba)
            row.update({"experiment": name, "fold": fold})
            cv_rows.append(row)
            fold_models.append(model)
            high_recall = row.get("High_recall", np.nan)
            print(
                f"  fold {fold}: macro_f1={row['macro_f1']:.5f}, "
                f"log_loss={row['log_loss']:.5f}, "
                f"High_recall={high_recall:.5f}"
            )

            del train_pool, valid_pool, model
            gc.collect()

        oof_predictions[name] = oof_pred
        oof_probabilities[name] = oof_proba
        trained_fold_models[name] = fold_models

    cv_results = pd.DataFrame(cv_rows)
    display(cv_results)
    cv_summary = summarize_cv(cv_results)
    display(cv_summary)
else:
    cv_results = pd.DataFrame()
    cv_summary = pd.DataFrame()
    print("Cross-validation disabled. Set RUN_FULL_CV = True to run experiments.")

## 7. Experiment Diagnostics

Inspect the best experiment from cross-validation. Focus on macro F1, log loss, and the `High` class precision/recall tradeoff.

In [ ]:
if RUN_FULL_CV and not cv_results.empty:
    best_experiment = (
        cv_results.groupby("experiment")["macro_f1"]
        .mean()
        .sort_values(ascending=False)
        .index[0]
    )
    print("Best experiment by mean macro F1:", best_experiment)
    print(
        classification_report(
            y, oof_predictions[best_experiment], target_names=class_names, digits=4
        )
    )
    plot_confusion(y, oof_predictions[best_experiment], f"OOF {best_experiment}")

    metric_view = cv_results[
        ["experiment", "fold", "macro_f1", "log_loss", "balanced_accuracy"]
        + [f"{label}_recall" for label in class_names]
    ]
    display(metric_view.sort_values(["experiment", "fold"]))
else:
    best_experiment = EXPERIMENTS[0]["experiment"]
    print(
        "Using default best experiment because CV results are not available:",
        best_experiment,
    )

## 8. Feature Importance

Fit the selected configuration on a stratified holdout split for feature interpretation. This confirms whether tuning still relies on the same agronomic drivers identified by EDA and the baseline notebook.

In [ ]:
selected_exp = next(e for e in EXPERIMENTS if e["experiment"] == best_experiment)
selected_df = selected_exp["train_df"]
selected_features = selected_exp["feature_cols"]
selected_params = selected_exp["params"]

tr_idx, va_idx = train_test_split(
    np.arange(len(selected_df)),
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

holdout_model = CatBoostClassifier(**selected_params)
holdout_model.fit(
    make_pool(selected_df.iloc[tr_idx], selected_features, y[tr_idx]),
    eval_set=make_pool(selected_df.iloc[va_idx], selected_features, y[va_idx]),
    use_best_model=True,
)

importance = holdout_model.get_feature_importance(prettified=True).rename(
    columns={"Feature Id": "feature", "Importances": "importance"}
)
display(importance.head(30))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=importance.head(25),
    x="importance",
    y="feature",
    hue="feature",
    palette=VIRIDIS_CMAP,
    legend=False,
)
plt.title(f"Feature Importance - {best_experiment}")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 9. Final Training and Submission

Train the selected CatBoost configuration on the full training data and create a tuned `submission.csv`. Review the prediction distribution before submitting.

In [ ]:
if MAKE_SUBMISSION:
    final_model = CatBoostClassifier(**selected_params)
    final_model.fit(make_pool(selected_df, selected_features, y))

    selected_test_df = selected_exp["test_df"]
    test_pool = make_pool(selected_test_df, selected_features)
    test_pred = final_model.predict(test_pool)
    if test_pred.ndim > 1:
        test_pred = test_pred.reshape(-1)
    test_pred = test_pred.astype(int)
    test_labels = label_encoder.inverse_transform(test_pred)

    submission = sample_submission.copy()
    submission[target_col] = test_labels
    submission.to_csv("submission.csv", index=False)

    display(submission.head())
    display(
        submission[target_col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("prediction_pct")
        .to_frame()
    )
    print("Wrote submission.csv using:", best_experiment)
else:
    print("Submission creation disabled.")

## 10. Tuning Summary and Next Steps

Record the best cross-validation result, leaderboard score, selected parameters, and whether interaction or threshold features improved the model.

Current tuning interpretation:

- `baseline_interactions` was the best cross-validation experiment by mean macro F1: `0.96994`.
- `deeper_interactions` had slightly better mean log loss (`0.06016` vs `0.06088`) but did not improve macro F1.
- Threshold features did not improve performance over the interaction baseline.
- Class weighting improved `High` recall to about `0.946`, but it reduced macro F1 and worsened log loss.
- The validated public leaderboard score from the reused tuning submission is `0.96094`.

Recommended next experiment: use the current submission as the stable baseline, then test a compact ensemble or calibration workflow rather than rerunning a broad CatBoost grid.